# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umaimakhalid17/ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane, locked this week: Lane 2 — Refresh / Content Opportunity Scoring.** `w01` drafted Lane 2,
`w02`/`w03` explored Lane 3 (clustering). This week's deliverable — a rule with a score, one
reason code, an action label, and a ranked queue a human reviews top-to-bottom — is a
scoring/ranking task by definition; clustering has no natural "top 10," no action label, and
nothing for precision@K to grade. So I'm confirming **Lane 2** here, not switching casually: the
assignment itself only makes sense for a lane with a decision at the end. `w02`/`w03`'s clustering
work stays in the repo as exploration, not the locked lane.

## 1. My rule and its reason codes

**The rule, in plain words:** A page is worth an editor's review if it has real search demand
(enough impressions and a page-one-ish position that search users can actually see) **and** its
click-through rate is low for that position — that combination is the single strongest,
skeptic-tested signal I have (Signal 2 below). Staleness (how long since the page was last
updated) is the other signal FlyRank's own refresh flags lean on, so I test it first — and it
turns out to be a much weaker, non-monotonic signal in this data (Signal 1 below). Rather than
throw it out, I demote it: it no longer decides *who* gets flagged, only *which* action label a
flagged page gets (a full content refresh vs. a lighter CTR/meta review).

---

### Signal check 1 — staleness (behind the refresh flags: `stale_visible_page`)

**Claim being tested:** "The longer since a page was last updated, the more likely it's
currently declining" — this is the assumption baked into `stale_visible_page`
(`days_since_last_update >= 180` and `impressions_90d >= 500`).

**Test:** bucket every row by `days_since_last_update`, compare `is_declining_label` mean (the
`trend_direction == "down"` rate) and print `n` per bucket.

**Verdict: MIXED.** Decline rate rises from 51.1% (0–30 days) to 61.1% (91–180 days) — consistent
with the claim — but then **drops to 47.1%** in the 181+ bucket, the exact population the refresh
flag targets, and that bucket sits *below* the 54.2% overall base rate. `n=174` for 181+ clears
the ~50-row floor, so this isn't noise from a tiny cell — it's a real reversal at the range the
flag actually fires on. A clearly negative result: staleness alone does **not** cleanly predict
decline, so my rule won't use it to decide who's in the queue.

### Signal check 2 — CTR vs. position (behind the CTR-fix logic: `low_ctr_visible_page`)

**Claim being tested:** "CTR drops as position gets worse" — the assumption behind flagging a
page whose CTR is low *for its position* (`low_ctr_visible_page`: position 1–20, `ctr < 0.5`) as
an anomaly worth a human look, rather than flagging low CTR on its own.

**Test:** restrict to rows with real position data (`avg_position > 0`), bucket by
`position_tier`, and report both the mean of each page's own CTR and the impression-weighted CTR
(sum of clicks / sum of impressions — the auditing-signals skill's warning that averaging
per-row rates isn't the true rate, so I check both ways).

**Verdict: CONFIRMED.** CTR falls at every step from `top_3` (2.76% mean-of-row CTR, n=1,116)
through `page_1` (0.65%, n=11,814), `striking` (0.32%, n=7,304), `page_3_5` (0.22%, n=7,242), to
`deep` (0.15%, n=1,319) — monotonic, and the impression-weighted view (dominated by the handful
of giant pages) points the same direction. A page with `ctr < 0.5` at position 1–20 really is
sitting below where its position predicts it should be. This is the signal my rule leans on to
decide who's in the queue.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 140)

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("rows:", len(df), "| overall decline rate:", round(df["is_declining_label"].mean() * 100, 1), "%")
print()

# ---- Signal check 1: staleness vs. decline (behind stale_visible_page / refresh flags) ----
bins = [0, 30, 90, 180, 10_000]
labels = ["0-30", "31-90", "91-180", "181+"]
df["staleness_bucket"] = pd.cut(df["days_since_last_update"], bins=bins, labels=labels, right=True)

signal1_table = df.groupby("staleness_bucket", observed=True).agg(
    n=("content_id", "size"),
    decline_rate_pct=("is_declining_label", lambda s: round(s.mean() * 100, 1)),
)
print("Signal 1 -- staleness vs. decline rate (n printed):")
print(signal1_table)
print("verdict: MIXED -- rises 0-30 -> 91-180, then reverses at 181+ (the flag's own target range)")
print()

# ---- Signal check 2: CTR vs. position (behind low_ctr_visible_page / CTR-fix logic) ----
d = df[df["avg_position"] > 0].copy()  # avg_position == 0 means "no data", not rank zero
tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]

signal2_table = d.groupby("position_tier", observed=True).apply(
    lambda g: pd.Series({
        "n": len(g),
        "mean_of_row_ctr_pct": round(g["ctr"].mean(), 2),
        "weighted_ctr_pct": round(100 * g["clicks_90d"].sum() / g["impressions_90d"].sum(), 2),
    })
).reindex(tier_order)
print("Signal 2 -- CTR by position tier (n printed):")
print(signal2_table)
print("verdict: CONFIRMED -- CTR falls monotonically as position tier worsens, both ways of measuring it")


rows: 30000 | overall decline rate: 54.2 %

Signal 1 -- staleness vs. decline rate (n printed):
                      n  decline_rate_pct
staleness_bucket                         
0-30              20480              51.1
31-90               175              58.9
91-180             9171              61.1
181+                174              47.1
verdict: MIXED -- rises 0-30 -> 91-180, then reverses at 181+ (the flag's own target range)

Signal 2 -- CTR by position tier (n printed):
                     n  mean_of_row_ctr_pct  weighted_ctr_pct
position_tier                                                
top_3           1116.0                 2.76              0.49
page_1         11814.0                 0.65              0.35
striking        7304.0                 0.32              0.35
page_3_5        7242.0                 0.22              0.15
deep            1319.0                 0.15              0.04
verdict: CONFIRMED -- CTR falls monotonically as position tier worsens, both wa

## 2. Build the ranked queue (writes the CSV)

**The score, coded transparently (no fitted weights):**

```text
eligible   = impressions_90d >= 500  AND  0 < avg_position <= 20      # real, visible demand
ctr_gap    = eligible AND ctr < 0.5                                    # Signal 2's confirmed anomaly
severity   = max(0, 0.5 - ctr)                                         # how far below the 0.5% bar
action_score = log1p(impressions_90d) * severity   (0 when not ctr_gap)
```

`log1p(impressions_90d)` keeps the score from being dominated purely by a few giant pages
(heavy-tail handling, same idea the auditing-signals skill flags). Staleness — demoted after the
MIXED verdict — decides only which of two reason codes a `ctr_gap` page gets, not whether it's in
the queue at all.

**Reason code (exactly one per row) and the action label it maps to:**

| `reason_code` | condition | `action_label` |
|---|---|---|
| `low_ctr_and_stale` | `ctr_gap` AND `days_since_last_update >= 180` | `refresh_content_and_ctr` |
| `low_ctr_visible_page` | `ctr_gap` AND not stale | `review_ctr_and_meta` |
| `not_flagged` | neither | `monitor` |

In [2]:
import os

eligible = (df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)
ctr_gap = eligible & (df["ctr"] < 0.5)
stale = df["days_since_last_update"] >= 180

df["reason_code"] = np.select(
    [ctr_gap & stale, ctr_gap & ~stale],
    ["low_ctr_and_stale", "low_ctr_visible_page"],
    default="not_flagged",
)

severity = (0.5 - df["ctr"]).clip(lower=0)
df["action_score"] = np.where(ctr_gap, np.log1p(df["impressions_90d"]) * severity, 0.0)

action_map = {
    "low_ctr_and_stale": "refresh_content_and_ctr",
    "low_ctr_visible_page": "review_ctr_and_meta",
    "not_flagged": "monitor",
}
df["action_label"] = df["reason_code"].map(action_map)

queue = df.sort_values("action_score", ascending=False).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)

print("reason_code counts:")
print(queue["reason_code"].value_counts())
print()
print("decline rate by reason_code (lift check vs. the 54.2% base rate):")
print(queue.groupby("reason_code")["is_declining_label"].mean().round(3))
print()

output_cols = [
    "rank", "content_id", "client_id", "action_score", "reason_code", "action_label",
    "impressions_90d", "avg_position", "ctr", "days_since_last_update", "word_count",
    "content_type", "is_declining_label",
]

os.makedirs("../outputs", exist_ok=True)
out_path = "../outputs/baseline_action_score.csv"
queue[output_cols].to_csv(out_path, index=False)
print(f"Wrote ranked queue: {out_path}  ({len(queue)} rows)")

import json
metrics = {
    "rows": int(len(queue)),
    "n_ctr_gap_flagged": int((queue["reason_code"] != "not_flagged").sum()),
    "n_low_ctr_and_stale": int((queue["reason_code"] == "low_ctr_and_stale").sum()),
    "decline_rate_overall": round(float(df["is_declining_label"].mean()), 4),
    "decline_rate_by_reason_code": {
        k: round(float(v), 4)
        for k, v in queue.groupby("reason_code")["is_declining_label"].mean().items()
    },
    "signal1_staleness_verdict": "MIXED",
    "signal2_ctr_vs_position_verdict": "CONFIRMED",
    "score_formula": "log1p(impressions_90d) * max(0, 0.5 - ctr), 0 if not (visible & position<=20 & ctr<0.5)",
}
with open("../outputs/baseline_metadata.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Wrote metrics: ../outputs/baseline_metadata.json")


reason_code counts:
reason_code
not_flagged             20241
low_ctr_visible_page     9749
low_ctr_and_stale          10
Name: count, dtype: int64

decline rate by reason_code (lift check vs. the 54.2% base rate):
reason_code
low_ctr_and_stale       1.000
low_ctr_visible_page    0.627
not_flagged             0.501
Name: is_declining_label, dtype: float64



Wrote ranked queue: ../outputs/baseline_action_score.csv  (30000 rows)
Wrote metrics: ../outputs/baseline_metadata.json


## 3. Top-10 review

For each of the top ten by `action_score`: the action, why it's there, and what would make the
call wrong.

| Rank | content_id | Action | Why it's there | What would make it wrong |
|---|---|---|---|---|
| 1 | `content_c8e9d6ab9013` | review_ctr_and_meta | 208,678 impressions, position 9.7, **ctr 0.00%** — visible page, essentially zero clicks | If the title/meta shown in search is already fine and the real cause is a SERP feature (featured snippet, ads) stealing the click above it — no amount of on-page editing fixes that |
| 2 | `content_8451fc6f034d` | review_ctr_and_meta | 272,144 impressions, position 2.3, ctr 0.03% — near-top-3 position with almost no clicks, the widest CTR gap in the queue | If position 2.3 is a rounding artifact of a page that flips between #1 and #4 across the 90 days, the "should be getting way more clicks" story is weaker than a stable #2 |
| 3 | `content_453722754fea` | review_ctr_and_meta | 140,079 impressions, position 7.6, ctr 0.01% | If `main_intent` is navigational (a branded query), low CTR may just mean users already know the URL and skip the snippet — not a meta-copy problem |
| 4 | `content_c84a0ab98e90` | review_ctr_and_meta | 223,271 impressions, position 7.8, ctr 0.03% | Same client as #3 (`client_f369cb89fc`) — if this is a client-wide tracking or tagging issue rather than a per-page copy problem, editing individual titles won't help |
| 5 | `content_4a6607efcb46` | review_ctr_and_meta | 128,068 impressions, position 2.2, ctr 0.01%, 4,939-word page | A very long page ranking well with near-zero CTR is unusual enough that I'd check for a tracking/measurement bug before assuming the meta copy is the problem |
| 6 | `content_39881853ef0c` | review_ctr_and_meta | 112,434 impressions, position 7.2, ctr 0.01%, same client again | Third page from `client_f369cb89fc` in the top 6 — reinforces the client-level-issue possibility from #4 rather than three independent content problems |
| 7 | `content_36ff89c8214e` | review_ctr_and_meta | 295,097 impressions, position 7.3, ctr 0.05%, `word_count` missing | Missing word count means I can't rule out this is a `feedly article`-style page where thinness, not the title, is the real issue — the reason code doesn't capture that |
| 8 | `content_0919dd345d80` | review_ctr_and_meta | 119,217 impressions, position 7.0, ctr 0.02%, updated 7 days ago | Freshly updated and still at ctr 0.02% — if a very recent update already tried to fix this and it didn't move the needle, "review the meta again" may not be the right next action |
| 9 | `content_c1fe78bc4e37` | review_ctr_and_meta | 134,055 impressions, position 7.5, ctr 0.03%, `word_count` missing | Same missing-word-count gap as #7 — worth checking `content_type` before assuming a meta fix is enough |
| 10 | `content_b115f7c74779` | review_ctr_and_meta | 123,469 impressions, position 8.0, ctr 0.03% | Fourth page from `client_19581e27de` across the top 10 (with #1, #7, #9) — before treating this as ten independent picks, worth checking whether one client's tracking setup is driving several of them |

In [3]:
top10 = queue.head(10)[output_cols]
print(top10.to_string(index=False))
print()
print("Every top-10 row shares reason_code == 'low_ctr_visible_page' (none crossed the")
print("staleness threshold too) -- action_score is dominated by CTR severity at high impression")
print("volume, which is exactly what the score formula rewards. Two clients")
print("(client_f369cb89fc, client_19581e27de) each supply 3-4 of the top 10 -- flagged in the")
print("review above as a reason to check for a client-level cause before treating these as ten")
print("independent content problems.")


 rank           content_id         client_id  action_score          reason_code        action_label  impressions_90d  avg_position  ctr  days_since_last_update  word_count    content_type  is_declining_label
    1 content_c8e9d6ab9013 client_19581e27de      6.124276 low_ctr_visible_page review_ctr_and_meta           208678           9.7 0.00                     104         NaN keyword article                   1
    2 content_8451fc6f034d client_d029fa3a95      5.881622 low_ctr_visible_page review_ctr_and_meta           272144           2.3 0.03                      20      3528.0 keyword article                   0
    3 content_453722754fea client_f369cb89fc      5.806485 low_ctr_visible_page review_ctr_and_meta           140079           7.6 0.01                      20      2700.0 keyword article                   1
    4 content_c84a0ab98e90 client_f369cb89fc      5.788589 low_ctr_visible_page review_ctr_and_meta           223271           7.8 0.03                      20      287

## 4. Weak picks + leakage check

**Weakest picks, and why:** #1, #5, and #8 are the shakiest of the ten. #1 and #5 have `ctr`
values so close to exactly 0.00%/0.01% on six-figure impression counts that a tracking or
tagging bug is at least as plausible an explanation as a genuine content problem — I have no
column that distinguishes "nobody clicks this" from "clicks aren't being measured." #8 was
updated only 7 days ago and still scores near the top, which undercuts the story that a fresh
review will move it — if an editor already looked at it recently and CTR didn't respond, my
queue is asking them to look again with no new information. More broadly: 3–4 of the top 10 come
from just two client_ids, which the score formula has no way to notice or downweight — a
client-level issue could masquerade as several independent "content problems."

**Leakage check.** The rule uses only `impressions_90d`, `avg_position`, `ctr`, and
`days_since_last_update` — all observed, current-window signals, none of them derived from
`trend_direction`/`trend_pct` (the label source) and none of them a FlyRank product decision
flag (`health_score`, `priority_score`, `action_type` are not in this dataset at all, per the
lane guide). `is_declining_label` appears in the output **only** as a column to eyeball lift
against — never as an input to `action_score`, `reason_code`, or `action_label`. No future
window is used: everything is a trailing-90-day or "as of today" measurement, and the starter
CSV has no forward-looking columns to accidentally reach into.

In [4]:
score_inputs = {"impressions_90d", "avg_position", "ctr", "days_since_last_update"}
label_derived = {"trend_direction", "trend_pct", "is_declining_label"}
product_flags_not_in_dataset = {"health_score", "priority_score", "action_type", "refresh_tier"}

print("Inputs actually used to build action_score / reason_code:", score_inputs)
print("Overlap with label-derived columns (must be empty):", score_inputs & label_derived)
print("Overlap with product decision flags (must be empty -- and these columns don't exist here):",
      score_inputs & set(df.columns) & product_flags_not_in_dataset)
print("is_declining_label used only for the lift check in Section 2/3, never inside the score:",
      "is_declining_label" not in score_inputs)

client_counts = queue.head(10)["client_id"].value_counts()
print()
print("client_id concentration in the top 10 (flagged as a weak-pick pattern above):")
print(client_counts)


Inputs actually used to build action_score / reason_code: {'ctr', 'avg_position', 'days_since_last_update', 'impressions_90d'}
Overlap with label-derived columns (must be empty): set()
Overlap with product decision flags (must be empty -- and these columns don't exist here): set()
is_declining_label used only for the lift check in Section 2/3, never inside the score: True

client_id concentration in the top 10 (flagged as a weak-pick pattern above):
client_id
client_19581e27de    4
client_f369cb89fc    3
client_d029fa3a95    1
client_6208ef0f77    1
client_4e07408562    1
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.